# Tratamento de dados e pré-separação

In [1]:
import pandas as pd

caminho_csv = '../data/oncological_panel_datasus.csv'

df = pd.read_csv(caminho_csv)

# Conferindo número de linhas e colunas da base de dados
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.\n")
display(df.head())

O dataset possui 995374 linhas e 24 colunas.



,Unnamed: 0,ANO_DIAGN,ANOMES_DIA,ANO_TRATAM,ANOMES_TRA,UF_RESID,MUN_RESID,UF_TRATAM,MUN_TRATAM,UF_DIAGN,...,SEXO,ESTADIAM,CNES_DIAG,CNES_TRAT,TEMPO_TRAT,CNS_PAC,DIAG_DETH,DT_DIAG,DT_TRAT,DT_NASC
0,0,2013,201304,2014.0,201401.0,35,354100,35.0,354850.0,35,...,F,2.0,2079720,2079720.0,268.0,NaN,C50,2013-11-04,2014-04-01,1963-03-07
1,1,2013,201306,2013.0,201309.0,43,430090,43.0,430700.0,43,...,M,1.0,2707918,2707918.0,92.0,NaN,C20,2013-06-17,2013-09-17,1958-10-28
2,2,2013,201308,2013.0,201310.0,35,355030,35.0,355030.0,35,...,F,0.0,2078287,2080575.0,63.0,NaN,C50,2013-08-20,2013-10-22,1961-01-06
3,3,2013,201303,2013.0,201303.0,26,261160,26.0,261160.0,26,...,F,5.0,582,582.0,-15.0,NaN,C54,2013-03-22,2013-07-03,1960-02-07
4,4,2013,201304,2013.0,201304.0,42,420450,42.0,420910.0,42,...,M,NaN,2436469,2436469.0,3.0,NaN,C91,2013-09-04,2013-12-04,1988-10-07


1. Limpando a coluna de tempo de tratamento 

In [2]:
import numpy as np 

# Transformando todas as informações em numérico 
df['TEMPO_TRAT'] = pd.to_numeric(df['TEMPO_TRAT'], errors='coerce')

# Transformando códigos de erro em nulos
df.loc[df['TEMPO_TRAT']> 5000, 'TEMPO_TRAT'] = np.nan

# Quantidade de nulos antes da limpeza
nulos_antes = df['TEMPO_TRAT'].isna().sum()

print(f"Valores nulos em 'TEMPO_TRAT' antes da limpeza: {nulos_antes}")

# Removendo linhas sem informação de tempo de tratamento
df = df.dropna(subset=['TEMPO_TRAT']).copy()

print(f"Registro após remover nulos: {len(df)}")

Valores nulos em 'TEMPO_TRAT' antes da limpeza: 411428
Registro após remover nulos: 583946


Observação: Valores nulos marcados como 0, porém, o dataframe caiu de 995.374 para 583.946 linhas, uma redução de quase 41%. O que indica que os dados faltantes não estavam registrados como nulos oficiais do python (NaN) e também registros estavam preenchidos com códigos de erro do DataSUS.

2. Criação da variável-alvo binária ('ATRASO_60_DIAS')

In [3]:
# Criando a variável
df['ATRASO_60_DIAS'] = (df['TEMPO_TRAT'] > 60).astype(int)

# Verificando a nova variável
print(df['ATRASO_60_DIAS'].value_counts(normalize=True).round(4) * 100)

ATRASO_60_DIAS
0    56.67
1    43.33
Name: proportion, dtype: float64


0 = Pacientes que iniciaram o tratamento em tempo <= 60 dias (prazo)
1 = Pacientes que iniciaram o tratamento em tempo > 60 dias (atraso)

3. Conversão das datas para o tipo nativo datetime

In [4]:
# Convertendo data do diagnóstico
df['DT_DIAG'] = pd.to_datetime(df['DT_DIAG'], format='%Y-%m-%d', errors='coerce')

# Convertendo ano e mês do diagnóstico
df['ANOMES_DIA'] = pd.to_datetime(df['ANOMES_DIA'].astype(str).str.slice(0, 6), format='%Y%m', errors='coerce')

# Removendo linhas que não possuem a referência de data que usaremos na separaçã
df = df.dropna(subset=['ANOMES_DIA'])

4. Validação final 

In [5]:
# Conferindo tipos e nulos
print("Colunas tratadas")
df[['TEMPO_TRAT', 'ATRASO_60_DIAS', 'DT_DIAG', 'ANOMES_DIA']].info()

# Conferindo amostra aleatória
display(df[['TEMPO_TRAT', 'ATRASO_60_DIAS', 'DT_DIAG', 'ANOMES_DIA']].sample(5))

Colunas tratadas
<class 'pandas.DataFrame'>
Index: 583946 entries, 0 to 995371
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   TEMPO_TRAT      583946 non-null  float64       
 1   ATRASO_60_DIAS  583946 non-null  int64         
 2   DT_DIAG         583946 non-null  datetime64[us]
 3   ANOMES_DIA      583946 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(1), int64(1)
memory usage: 22.3 MB


,TEMPO_TRAT,ATRASO_60_DIAS,DT_DIAG,ANOMES_DIA
970435,14.0,0,2024-04-18,2024-04-01
855826,47.0,0,2024-11-21,2024-11-01
799207,61.0,1,2023-01-10,2023-10-01
314702,67.0,1,2019-07-02,2019-02-01
23708,16.0,0,2013-05-29,2013-05-01


5. Exportação da base limpa 

In [ ]:
# Base tratada para próximas etapas
caminho_salvamento = '../data/oncological_panel_limpo.csv'

df.to_csv(caminho_salvamento, index=False)

print(f"Base limpa salva com sucesso em: {caminho_salvamento}")